In [23]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")

import warnings
warnings.filterwarnings('ignore')

In [31]:
train_data_path = "/Users/hongpite/Desktop/就业/kaggle/Titanic_data/train.csv"
test_data_path = "/Users/hongpite/Desktop/就业/kaggle/Titanic_data/test.csv"
gender_test_path = "/Users/hongpite/Desktop/就业/kaggle/Titanic_data/gender_submission.csv"

df_train = pd.read_csv(train_data_path)
df_test = pd.read_csv(test_data_path)
df_gender = pd.read_csv(gender_test_path)

df_all = pd.concat([df_train, df_test], sort = True).reset_index(drop = True)
# 上下拼接
# sort =True 列按照列字母自动排序
# reset_index 生成新索引，drop = True删除旧索引， =False 吧旧索引存入一个叫index的新列

print('Number of Training Examples = {}'.format(df_train.shape[0]))
print('Number of Test Examples = {}\n'.format(df_test.shape[0]))
print('Training Shape = {}'.format(df_train.shape))

print('Test Shape = {}'.format(df_test.shape))

print(df_train.columns)
print(df_test.columns)

print("The info of df_train: ")
print(df_train.info())
print(df_train.sample(3))

print("The info of df_test: ")
print(df_test.info())
print(df_test.sample(3))

Number of Training Examples = 891
Number of Test Examples = 418

Training Shape = (891, 12)
Test Shape = (418, 11)
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
The info of df_train: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 

In [ ]:
df_train.name = "Training Set"
df_test.name = "Test Set"
dfs = [df_train, df_test]
# 只看属性名
# df_train.__dict__
# [v for v in dir(df_train) if not v.startswith('_')]
# df_train.name 是一个属性，它存在了df_train这个变量里，但是没有存在数据表里，不会被导入csv中
# df["col"]操作的是数据，df.col是语法糖，应该少用。创建属性可以使用df.name。
# 属性是python对象本身的操作，与数据无关

In [32]:
# 检测出缺失值
def display_missing(df):
    for col in df.columns.tolist():
        print("{} column missing values: {}".format(col, df[col].isna().sum()))
    print("\n")

for df in dfs:
    print("{}".format(df.name))
    display_missing(df)

Training Set
PassengerId column missing values: 0
Survived column missing values: 0
Pclass column missing values: 0
Name column missing values: 0
Sex column missing values: 0
Age column missing values: 177
SibSp column missing values: 0
Parch column missing values: 0
Ticket column missing values: 0
Fare column missing values: 0
Cabin column missing values: 687
Embarked column missing values: 2


Test Set
PassengerId column missing values: 0
Pclass column missing values: 0
Name column missing values: 0
Sex column missing values: 0
Age column missing values: 86
SibSp column missing values: 0
Parch column missing values: 0
Ticket column missing values: 0
Fare column missing values: 1
Cabin column missing values: 327
Embarked column missing values: 0




In [ ]:
#处理Age的缺失值
# 方法一，直接删除缺失值
# 方法二，填补缺失值，可以填入中位数，全局中位数
# 按照特征的分组填入中位数，至于选择什么分组，可以看Age和某个特征的具体的相关性，比如按照Sex和Pclass
# 可以考虑Title分组，可能比Sex更细化。而且一个人的Title与年龄也是有一定相关性的。但是Title里面过于细碎，有时候要合并一些Title的分组
# 缺失值也是一种信息。1.随机缺失，2.某些特征的更容易缺失
# 增加缺失指示变量 Age_missing = 1表示原始Age缺失，0表示原始Age没缺失
# 还可以用其他变量，再训练一个回归模型来预测Age
# 或者对于某些模型，可以直接处理缺失值，例如LightGBM

# 考虑test和train，用全部的数据集来填补Age。

In [33]:
df_all.corr()

ValueError: could not convert string to float: 'C85'

In [48]:
df_all_corr = df_all.corr(numeric_only=True).abs().unstack().sort_values(kind="quicksort", ascending=False).reset_index()
# df.corr() 对于df的所有列，两两计算相关性，生成相关系数矩阵,返回的是df格式
# df.unstack 把矩形变成列输出 返回的是Series
# sort_values(ascending=False) 对于Series的值降序排列
df_all_corr.rename(columns={"level_0": "Feature 1", "level_1": "Feature 2", 0: 'Correlation Coefficient'}, inplace=True)


df_all_corr[df_all_corr['Feature 1'] == 'Age']

,Feature 1,Feature 2,Correlation Coefficient
0,Age,Age,1.000000
9,Age,Pclass,0.408106
18,Age,SibSp,0.243699
21,Age,Fare,0.178740
26,Age,Parch,0.150917
30,Age,Survived,0.077221
41,Age,PassengerId,0.028814


In [55]:
Series_test = df_all.corr(numeric_only=True).unstack().sort_values()
# unstack 之后，就变成了多层索引
print(Series_test.index)
print(Series_test.index.nlevels)

MultiIndex([(       'Fare',      'Pclass'),
            (     'Pclass',        'Fare'),
            (        'Age',      'Pclass'),
            (     'Pclass',         'Age'),
            (     'Pclass',    'Survived'),
            (   'Survived',      'Pclass'),
            (      'SibSp',         'Age'),
            (        'Age',       'SibSp'),
            (      'Parch',         'Age'),
            (        'Age',       'Parch'),
            (   'Survived',         'Age'),
            (        'Age',    'Survived'),
            (      'SibSp', 'PassengerId'),
            ('PassengerId',       'SibSp'),
            (     'Pclass', 'PassengerId'),
            ('PassengerId',      'Pclass'),
            (   'Survived',       'SibSp'),
            (      'SibSp',    'Survived'),
            ('PassengerId',    'Survived'),
            (   'Survived', 'PassengerId'),
            (      'Parch', 'PassengerId'),
            ('PassengerId',       'Parch'),
            (      'Parch',     